In [18]:
import pandas as pd
import numpy as np
import torch.nn as nn
# Chargement
df = pd.read_csv('creditcard.csv')

# Exploration
print("Shape :", df.shape)
print("\nColonnes :", df.columns.tolist())
print("\nAperçu :")
print(df.head())
print("\nDistribution des classes :")
print(df['Class'].value_counts())
print(f"\nPourcentage de fraudes : {df['Class'].mean():.4%}")
print(f"\nValeurs manquantes : {df.isnull().sum().sum()}")


from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader, TensorDataset

# 1. Séparer X et y
X = df.drop('Class', axis=1).values
y = df['Class'].values

# 2. Normaliser
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# 4. Convertir en tenseurs
X_train_py = torch.FloatTensor(X_train)
X_test_py  = torch.FloatTensor(X_test)
y_train_py = torch.FloatTensor(y_train).unsqueeze(1)
y_test_py  = torch.FloatTensor(y_test).unsqueeze(1)

# 5. DataLoaders
train_loader = DataLoader(TensorDataset(X_train_py, y_train_py),
                          batch_size=32, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_py, y_test_py),
                          batch_size=32, shuffle=False)

print(f"X_train : {X_train_py.shape}")
print(f"X_test  : {X_test_py.shape}")
print(f"y_train : {y_train_py.shape}")
print(f"y_test  : {y_test_py.shape}")

#MODELE 
model=nn.Sequential(
    nn.Linear(30,60),
    nn.ReLU(),
    nn.Linear(60,15),
    nn.ReLU(),
    nn.Linear(15,1),
    nn.Sigmoid()
)

print(model)
total = sum(p.numel() for p in model.parameters())
print(f"Paramètres totaux : {total:,}")

import torch.nn as nn
import torch

loss_fn    = nn.BCELoss()
optimiseur = torch.optim.Adam(model.parameters(), lr=0.001)

def entrainer(modele, loader, loss_fn, optimiseur):
    model.train()
    total_loss=0

    for X_batch,y_batch in loader:
        optimiseur.zero_grad()
        predictions=model(X_batch)
        loss=loss_fn(predictions,y_batch)
        loss.backward()
        optimiseur.step()

        total_loss+=loss.item()
    return total_loss/len(loader)
    
def evaluer(modele, loader, loss_fn):
    model.eval()
    total_loss=0
    correct=0
    with torch.no_grad():   
        for X_batch, y_batch in loader:

            predictions = modele(X_batch)
            loss        = loss_fn(predictions, y_batch)
            total_loss += loss.item()

            pred_classes = (predictions > 0.5).float()
            correct += (pred_classes == y_batch).sum().item()

    accuracy = correct / len(loader.dataset)
    return total_loss / len(loader), accuracy

        
for epoch in range(20):
    moy_loss_train=entrainer(model,train_loader,loss_fn,optimiseur)
    moy_loss_test,acc=evaluer(model,test_loader,loss_fn)
    if epoch % 5 == 0:
        print(f"Epoch:{epoch:3d} |"
              f"Train Loss:{moy_loss_train:.4f} |"
              f"Test Loss:{moy_loss_test:.4f} |"
               f"Accuracy:{acc:.2%}")


Shape : (284807, 31)

Colonnes : ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']

Aperçu :
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.

In [19]:
# Installation
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Avant SMOTE : {sum(y_train==0)} légitimes | {sum(y_train==1)} fraudes")
print(f"Après SMOTE : {sum(y_train_sm==0)} légitimes | {sum(y_train_sm==1)} fraudes")


Avant SMOTE : 227451 légitimes | 394 fraudes
Après SMOTE : 227451 légitimes | 227451 fraudes


In [20]:
# Convertir en tenseurs PyTorch
X_train_sm_py = torch.FloatTensor(X_train_sm)
y_train_sm_py = torch.FloatTensor(y_train_sm).unsqueeze(1)

# Nouveau DataLoader avec données équilibrées
train_loader_sm = DataLoader(
    TensorDataset(X_train_sm_py, y_train_sm_py),
    batch_size=32,
    shuffle=True
)

print(f"X_train SMOTE : {X_train_sm_py.shape}")
print(f"y_train SMOTE : {y_train_sm_py.shape}")

X_train SMOTE : torch.Size([454902, 30])
y_train SMOTE : torch.Size([454902, 1])


In [21]:
from sklearn.metrics import classification_report, confusion_matrix

# Nouveau modèle (réinitialise les poids !)
modele2 = nn.Sequential(
    nn.Linear(30, 60),
    nn.ReLU(),
    nn.Linear(60, 15),
    nn.ReLU(),
    nn.Linear(15, 1),
    nn.Sigmoid()
)

loss_fn    = nn.BCELoss()
optimiseur = torch.optim.Adam(modele2.parameters(), lr=0.001)

# Entraînement sur données SMOTE
for epoch in range(20):
    modele2.train()
    total_loss = 0

    for X_batch, y_batch in train_loader_sm:
        optimiseur.zero_grad()
        preds = modele2(X_batch)
        loss  = loss_fn(preds, y_batch)
        loss.backward()
        optimiseur.step()
        total_loss += loss.item()

    if epoch % 5 == 0:
        # Évaluation sur test (données RÉELLES non SMOTE !)
        modele2.eval()
        with torch.no_grad():
            preds_test  = modele2(X_test_py)
            classes     = (preds_test > 0.5).float()
            acc         = (classes == y_test_py).float().mean()
            print(f"Epoch {epoch:3d} | "
                  f"Train Loss: {total_loss/len(train_loader_sm):.4f} | "
                  f"Accuracy: {acc:.2%}")

# ============================================
# MÉTRIQUES AVANCÉES
# ============================================
modele2.eval()
with torch.no_grad():
    preds_finales = modele2(X_test_py)
    classes_finales = (preds_finales > 0.5).float().numpy()

print("\n📊 RAPPORT COMPLET :")
print(classification_report(y_test_py.numpy(),
                            classes_finales,
                            target_names=['Légitime', 'Fraude']))

print("\n🔍 MATRICE DE CONFUSION :")
print(confusion_matrix(y_test_py.numpy(), classes_finales))

Epoch   0 | Train Loss: 0.0289 | Accuracy: 99.85%
Epoch   5 | Train Loss: 0.0155 | Accuracy: 99.90%
Epoch  10 | Train Loss: 0.0145 | Accuracy: 99.88%
Epoch  15 | Train Loss: 0.0143 | Accuracy: 99.91%

📊 RAPPORT COMPLET :
              precision    recall  f1-score   support

    Légitime       1.00      1.00      1.00     56864
      Fraude       0.74      0.86      0.79        98

    accuracy                           1.00     56962
   macro avg       0.87      0.93      0.90     56962
weighted avg       1.00      1.00      1.00     56962


🔍 MATRICE DE CONFUSION :
[[56834    30]
 [   14    84]]
